# 第17章：多因子选股策略实战

## 本章学习目标

- 完整实现选股策略
- 从数据处理到回测分析
- 理解完整工作流程
- 优化策略表现

---

## 17.1 项目概述

本项目将实现一个完整的多因子选股策略，涵盖从数据处理、特征工程、模型训练到回测分析的全流程。

### 项目流程

```
┌─────────────────────────────────────────────────────────────┐
│                    项目流程                                  │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 数据准备 ──→ 2. 特征工程 ──→ 3. 模型训练               │
│       ↓              ↓              ↓                       │
│  4. 模型评估 ──→ 5. 策略构建 ──→ 6. 回测分析               │
│       ↓              ↓              ↓                       │
│  7. 结果报告 ──→ 8. 优化建议 ──→ 9. 总结                    │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
import qlib
from qlib.data.dataset import DatasetH
from qlib.contrib.data.handler import Alpha158
from qlib.contrib.model.gbdt import LGBModel
from qlib.workflow import R
from qlib.data import D
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 17.2 数据准备

In [ ]:
# 配置参数
CONFIG = {
    "market": "csi300",
    "benchmark": "SH000300",
    "train_start": "2015-01-01",
    "train_end": "2018-12-31",
    "valid_start": "2019-01-01",
    "valid_end": "2020-06-30",
    "test_start": "2020-07-01",
    "test_end": "2022-12-31",
}

print("项目配置:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# 创建数据集
dataset = DatasetH(
    handler={
        "class": "Alpha158",
        "module_path": "qlib.contrib.data.handler",
        "kwargs": {
            "start_time": CONFIG["train_start"],
            "end_time": CONFIG["test_end"],
            "fit_start_time": CONFIG["train_start"],
            "fit_end_time": CONFIG["train_end"],
            "instruments": CONFIG["market"],
        },
    },
    segments={
        "train": (CONFIG["train_start"], CONFIG["train_end"]),
        "valid": (CONFIG["valid_start"], CONFIG["valid_end"]),
        "test": (CONFIG["test_start"], CONFIG["test_end"]),
    },
)

# 查看数据规模
train_data = dataset.prepare("train")
valid_data = dataset.prepare("valid")
test_data = dataset.prepare("test")

print(f"训练集: {train_data.shape}")
print(f"验证集: {valid_data.shape}")
print(f"测试集: {test_data.shape}")

## 17.3 模型训练

In [ ]:
# 创建模型
model = LGBModel(
    loss="mse",
    learning_rate=0.05,
    num_leaves=64,
    max_depth=6,
    n_estimators=500,
    colsample_bytree=0.8,
    subsample=0.8,
    random_state=42,
    n_jobs=4,
)

print("模型配置:")
print(f"  learning_rate: 0.05")
print(f"  num_leaves: 64")
print(f"  max_depth: 6")
print(f"  n_estimators: 500")

In [ ]:
# 使用 Recorder 记录实验
with R.start(experiment_name="multifactor_strategy") as recorder:
    
    # 记录配置
    recorder.log_params(CONFIG)
    
    # 训练模型
    print("开始训练模型...")
    model.fit(dataset)
    print("训练完成")
    
    # 保存模型
    recorder.save_object(model, name="model.pkl")
    
    print(f"\nRecorder ID: {recorder.id}")

## 17.4 模型评估

In [ ]:
# 预测
predictions = model.predict(dataset)

print(f"预测结果形状: {predictions.shape}")

In [ ]:
# 评估函数
def evaluate_model(predictions, labels):
    """评估模型"""
    pred = np.array(predictions).ravel()
    label = np.array(labels).ravel()
    
    mask = ~(np.isnan(pred) | np.isnan(label))
    pred = pred[mask]
    label = label[mask]
    
    # IC
    ic = np.corrcoef(pred, label)[0, 1]
    
    # Rank IC
    rank_ic = np.corrcoef(np.argsort(np.argsort(pred)), np.argsort(np.argsort(label)))[0, 1]
    
    return {"IC": ic, "Rank IC": rank_ic}

# 评估
test_data = dataset.prepare("test")
test_labels = test_data['label']

metrics = evaluate_model(predictions, test_labels)

print("模型评估结果:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# 计算每日 IC
def calculate_daily_ic(predictions, labels):
    """计算每日 IC"""
    dates = predictions.index.get_level_values('datetime').unique()
    
    ic_list = []
    for date in dates:
        pred_day = predictions.xs(date, level='datetime')
        label_day = labels.xs(date, level='datetime')
        
        common = pred_day.index.intersection(label_day.index)
        if len(common) > 10:
            ic = pred_day.loc[common].corr(label_day.loc[common])
            ic_list.append({'date': date, 'ic': ic})
    
    return pd.DataFrame(ic_list).set_index('date')

daily_ic = calculate_daily_ic(predictions, test_labels)

print(f"每日 IC 数量: {len(daily_ic)}")
print(f"平均 IC: {daily_ic['ic'].mean():.4f}")
print(f"ICIR: {daily_ic['ic'].mean() / daily_ic['ic'].std():.4f}")

In [ ]:
# 可视化 IC
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# IC 时序
axes[0].bar(daily_ic.index, daily_ic['ic'], 
            color=['green' if x > 0 else 'red' for x in daily_ic['ic']],
            alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].axhline(y=daily_ic['ic'].mean(), color='blue', linestyle='--', 
                label=f'平均 IC: {daily_ic["ic"].mean():.4f}')
axes[0].set_title('每日 IC 时序图')
axes[0].set_ylabel('IC')
axes[0].legend()

# IC 累计
axes[1].plot(daily_ic.index, daily_ic['ic'].cumsum(), linewidth=1.5)
axes[1].set_title('累计 IC')
axes[1].set_ylabel('累计 IC')

plt.tight_layout()
plt.show()

## 17.5 特征重要性分析

In [ ]:
# 获取特征重要性
importance = model.get_feature_importance()
importance_sorted = importance.sort_values(ascending=False)

print("特征重要性 Top 20:")
print(importance_sorted.head(20))

In [ ]:
# 可视化
plt.figure(figsize=(12, 8))

top_n = 30
importance_top = importance_sorted.head(top_n)

plt.barh(range(len(importance_top)), importance_top.values, color='steelblue')
plt.yticks(range(len(importance_top)), importance_top.index)
plt.xlabel('重要性分数')
plt.title('特征重要性 Top 30')
plt.tight_layout()
plt.show()

## 17.6 回测分析

In [ ]:
# 模拟回测
# 生成模拟收益数据
np.random.seed(42)

# 获取测试日期
test_dates = predictions.index.get_level_values('datetime').unique()
n_days = len(test_dates)

# 模拟策略收益（基于 IC）
base_return = 0.0005  # 日均收益
vol = 0.015  # 波动率
ic_factor = metrics['IC'] * 0.01  # IC 影响因子

strategy_returns = np.random.randn(n_days) * vol + base_return + ic_factor
benchmark_returns = np.random.randn(n_days) * vol + base_return

# 计算累计收益
strategy_cum = (1 + pd.Series(strategy_returns, index=test_dates)).cumprod()
benchmark_cum = (1 + pd.Series(benchmark_returns, index=test_dates)).cumprod()

print("模拟回测数据生成完成")

In [ ]:
# 可视化
plt.figure(figsize=(14, 6))

plt.plot(strategy_cum.index, strategy_cum.values, label='策略', linewidth=1.5)
plt.plot(benchmark_cum.index, benchmark_cum.values, label='基准', linewidth=1.5, linestyle='--')

plt.title('策略 vs 基准 累计收益')
plt.xlabel('日期')
plt.ylabel('累计收益')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 17.7 绩效报告

In [ ]:
# 计算绩效指标
def calculate_performance(returns, benchmark_returns=None):
    """计算绩效指标"""
    annual_return = (1 + returns.mean()) ** 252 - 1
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol
    
    cum = (1 + returns).cumprod()
    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max
    max_dd = drawdown.min()
    
    metrics = {
        "年化收益": f"{annual_return:.2%}",
        "年化波动率": f"{annual_vol:.2%}",
        "夏普比率": f"{sharpe:.2f}",
        "最大回撤": f"{max_dd:.2%}",
    }
    
    return metrics

strategy_metrics = calculate_performance(pd.Series(strategy_returns))

print("策略绩效指标:")
print("=" * 40)
for k, v in strategy_metrics.items():
    print(f"  {k}: {v}")

## 17.8 项目总结

In [ ]:
# 项目总结
print("项目总结:")
print("=" * 60)

print("\n1. 数据处理:")
print(f"   - 使用 Alpha158 特征集")
print(f"   - 训练集: {CONFIG['train_start']} ~ {CONFIG['train_end']}")
print(f"   - 测试集: {CONFIG['test_start']} ~ {CONFIG['test_end']}")

print("\n2. 模型表现:")
print(f"   - IC: {metrics['IC']:.4f}")
print(f"   - Rank IC: {metrics['Rank IC']:.4f}")
print(f"   - ICIR: {daily_ic['ic'].mean() / daily_ic['ic'].std():.4f}")

print("\n3. 优化建议:")
print("   - 尝试更多模型（LSTM、Transformer）")
print("   - 模型融合提升稳定性")
print("   - 调整 Topk 参数优化换手率")
print("   - 添加风控模块")

## 17.9 本章小结

本章我们完成了：

1. **数据准备**：配置时间范围、创建数据集
2. **模型训练**：使用 LightGBM 训练模型
3. **模型评估**：计算 IC、Rank IC、ICIR
4. **特征分析**：分析特征重要性
5. **回测模拟**：模拟策略表现
6. **绩效报告**：计算关键指标

### 关键收获

- 完整的选股策略流程
- 模型评估方法
- 回测分析技巧

### 下一章预告

下一章我们将实现行业轮动策略实战。